# Atividade Prática: Visualização Geoespacial com Folium

**Notebook completo com códigos e respostas no próprio arquivo.**

Objetivo: criar mapas interativos para analisar a distribuição de imóveis em **Nova Iguaçu** e **Queimados**, utilizando marcadores, `CircleMarker`, clustering e ícones personalizados.


## 1. Configuração do ambiente e base de dados

In [ ]:
import pandas as pd
import numpy as np
import folium
from folium.plugins import MarkerCluster

# Gerando dados sintéticos de imóveis em Nova Iguaçu e Queimados
np.random.seed(42)
n_imoveis = 45

# Coordenadas base aproximadas
# Nova Iguaçu: -22.756, -43.460
# Queimados: -22.716, -43.555

dados_imoveis = {
    'id_imovel': range(1, n_imoveis + 1),
    'cidade': np.where(np.random.rand(n_imoveis) > 0.4, 'Nova Iguaçu', 'Queimados'),
    'valor_venda': np.random.uniform(150000, 850000, n_imoveis).round(2),
    'tipo': np.random.choice(['Casa', 'Apartamento', 'Terreno'], n_imoveis)
}

df_mapa = pd.DataFrame(dados_imoveis)

def gerar_lat(cidade):
    if cidade == 'Nova Iguaçu':
        return -22.756 + np.random.uniform(-0.03, 0.03)
    return -22.716 + np.random.uniform(-0.02, 0.02)

def gerar_lon(cidade):
    if cidade == 'Nova Iguaçu':
        return -43.460 + np.random.uniform(-0.03, 0.03)
    return -43.555 + np.random.uniform(-0.02, 0.02)

df_mapa['latitude'] = df_mapa['cidade'].apply(gerar_lat)
df_mapa['longitude'] = df_mapa['cidade'].apply(gerar_lon)

df_mapa.head()


# Parte 1 – Inicialização e Marcadores Básicos

## 1. Mapa base centralizado na coordenada média

**Resposta:** A latitude e a longitude médias de todos os imóveis são calculadas e utilizadas como centro do mapa. O nível de zoom é definido como `12` e o estilo utilizado é o padrão `OpenStreetMap`.


In [ ]:
latitude_media = df_mapa['latitude'].mean()
longitude_media = df_mapa['longitude'].mean()

mapa_base = folium.Map(
    location=[latitude_media, longitude_media],
    zoom_start=12,
    tiles='OpenStreetMap'
)

mapa_base


## 2. Marcadores para os 5 primeiros imóveis

**Resposta:** Os cinco primeiros registros recebem um `folium.Marker`. O `popup` apresenta o tipo do imóvel e o valor de venda.


In [ ]:
for _, imovel in df_mapa.head(5).iterrows():
    popup = folium.Popup(
        f"<b>Tipo:</b> {imovel['tipo']}<br>"
        f"<b>Valor de venda:</b> R$ {imovel['valor_venda']:,.2f}",
        max_width=300
    )

    folium.Marker(
        location=[imovel['latitude'], imovel['longitude']],
        popup=popup
    ).add_to(mapa_base)

mapa_base


# Parte 2 – Customização Visual com Marcadores Circulares

## 3–4. `CircleMarker` para todos os imóveis

**Resposta:** Cada imóvel é representado por um círculo com raio fixo de **8 pixels**. A cor é definida conforme a cidade:
- **Azul:** Nova Iguaçu
- **Laranja:** Queimados

O texto **"Clique para detalhes"** aparece ao passar o mouse sobre o marcador.


In [ ]:
mapa_circulos = folium.Map(
    location=[latitude_media, longitude_media],
    zoom_start=12,
    tiles='OpenStreetMap'
)

cores_cidade = {
    'Nova Iguaçu': 'blue',
    'Queimados': 'orange'
}

for _, imovel in df_mapa.iterrows():
    cor = cores_cidade[imovel['cidade']]

    folium.CircleMarker(
        location=[imovel['latitude'], imovel['longitude']],
        radius=8,
        color=cor,
        fill=True,
        fill_color=cor,
        fill_opacity=0.7,
        tooltip='Clique para detalhes',
        popup=(
            f"<b>ID:</b> {imovel['id_imovel']}<br>"
            f"<b>Cidade:</b> {imovel['cidade']}<br>"
            f"<b>Tipo:</b> {imovel['tipo']}<br>"
            f"<b>Valor:</b> R$ {imovel['valor_venda']:,.2f}"
        )
    ).add_to(mapa_circulos)

mapa_circulos


# Parte 3 – Agrupamento Inteligente (Clustering)

## 5–6. `MarkerCluster` e ícones personalizados

**Resposta:** Para evitar excesso de marcadores próximos, é criado um `MarkerCluster`. Todos os 45 imóveis são adicionados ao cluster.

As cores dos ícones variam conforme o tipo:
- **Verde:** Casa
- **Azul:** Apartamento
- **Cinza:** Terreno


In [ ]:
mapa_cluster = folium.Map(
    location=[latitude_media, longitude_media],
    zoom_start=12,
    tiles='OpenStreetMap'
)

cluster = MarkerCluster().add_to(mapa_cluster)

cores_tipo = {
    'Casa': 'green',
    'Apartamento': 'blue',
    'Terreno': 'gray'
}

for _, imovel in df_mapa.iterrows():
    popup = folium.Popup(
        f"<b>ID:</b> {imovel['id_imovel']}<br>"
        f"<b>Cidade:</b> {imovel['cidade']}<br>"
        f"<b>Tipo:</b> {imovel['tipo']}<br>"
        f"<b>Valor de venda:</b> R$ {imovel['valor_venda']:,.2f}",
        max_width=300
    )

    marcador = folium.Marker(
        location=[imovel['latitude'], imovel['longitude']],
        popup=popup,
        tooltip='Clique para detalhes',
        icon=folium.Icon(
            color=cores_tipo[imovel['tipo']],
            icon='home' if imovel['tipo'] == 'Casa'
                  else 'building' if imovel['tipo'] == 'Apartamento'
                  else 'map-marker',
            prefix='glyphicon'
        )
    )

    marcador.add_to(cluster)

mapa_cluster


## 7. Salvando o mapa final

**Resposta:** O mapa com clustering e ícones personalizados é salvo como **`mapa_imoveis_baixada.html`**.


In [ ]:
mapa_cluster.save('mapa_imoveis_baixada.html')

print('Mapa salvo com sucesso como: mapa_imoveis_baixada.html')


# Conclusão

A atividade foi concluída utilizando **Folium** para criar uma visualização geoespacial interativa.

- O mapa base foi centralizado pela média das coordenadas dos imóveis.
- Os cinco primeiros imóveis foram apresentados com marcadores simples.
- Todos os imóveis foram representados com `CircleMarker`, diferenciando as cidades por cor.
- O `MarkerCluster` foi utilizado para reduzir a poluição visual causada pela concentração de pontos.
- Os ícones foram personalizados de acordo com o tipo do imóvel.
- O mapa final foi salvo no arquivo **`mapa_imoveis_baixada.html`**.

**Observação:** o arquivo HTML será criado pelo último código quando o notebook for executado no Google Colab.
